# Potato Blight Classification

**by Group 1**


---

## 1: Business Understanding

### 1.1 Business Context

Potato farming is one of the most economically significant agricultural activities in Sub-Saharan Africa, South Asia, and Latin America, where smallholder farmers depend on potato yields for both subsistence and income. However, potato crops are highly vulnerable to fungal diseases — most notably **Early Blight** (*Alternaria solani*) and **Late Blight** (*Phytophthora infestans*). Left undetected, these diseases can devastate 30–70% of a crop, causing severe food insecurity and financial loss.

Traditional disease identification relies on experts conducting in-person field inspections — a resource that most smallholder farmers cannot access in time. By the point visible symptoms have spread, the window for effective treatment is often already closed.

This project builds an **automated potato blight classification system** using the PlantVillage dataset and a MobileNetV2 deep learning architecture. The model classifies leaf images into three categories — **Early Blight**, **Late Blight**, and **Healthy** — enabling rapid, low-cost diagnosis directly from a smartphone photo.


### 1.2 Stakeholders

| Stakeholder | Interest |
|---|---|
| **Smallholder Farmers** | Early, accurate disease diagnosis to protect yield and income |
| **Agricultural Officers** | A scalable tool to assist field services |
| **NGOs & Food Security Bodies** | Reduction of crop loss in vulnerable communities |
| **Data / ML Teams** | Reproducible, maintainable model pipeline |

### 1.3 Project Plan (CRISP-DM Phases)

```
Phase 1: Business Understanding
Phase 2: Data Understanding
Phase 3: Data Preparation
Phase 4: Modeling
Phase 5: Evaluation
Phase 6: Deployment
```

**Deliverables:**
- Jupyter Notebook 
- GitHub Repository
- Deployment
- Non-Technical Presentation

---

## 2: Data Understanding

### 2.1 Overview — PlantVillage Potato Subset

The dataset is sourced from the **PlantVillage** project, a large-scale open dataset of plant leaf imagery developed at Penn State University. The full PlantVillage dataset covers 38 disease classes across 14 crop species; this project uses the **potato subset only**, consisting of 2,152 images across three classes.

| Class Folder | Disease | Images (approx.) | Share |
|---|---|---|---|
| `Potato___Early_blight` | *Alternaria solani* fungal infection | 1,000 | ~46 % |
| `Potato___Late_blight` | *Phytophthora infestans* water mould | 1,000 | ~46 % |
| `Potato___healthy` | No visible disease | 152 | ~7 % |



### 2.2 Import Libraries

In [1]:
# Imports

import os
import cv2
import PIL
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import (
    layers, Model, Sequential
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print(f'TensorFlow  : {tf.__version__}')
print(f'GPUs visible: {len(tf.config.list_physical_devices("GPU"))}')

TensorFlow  : 2.3.1
GPUs visible: 0


### 2.3 Configuration

In [ ]:
# for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Hyper-parameters 
IMG_SIZE    = 224          # MobileNetV2 expects 224×224
BATCH_SIZE  = 32
EPOCHS_HEAD = 15           # train only the new head
EPOCHS_FINE = 20           # fine-tune top layers of base
LR_HEAD     = 1e-3
LR_FINE     = 5e-5
UNFREEZE_FROM = 130        # unfreeze MobileNetV2 layers from this index onward

CLASS_NAMES = ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']
LABEL_MAP   = {c: i for i, c in enumerate(CLASS_NAMES)}
DISPLAY_NAMES = ['Early Blight', 'Late Blight', 'Healthy']


DATASET_DIR = 'PlantVillage'  
CHECKPOINT_PATH = 'best_potato_model.h5'